In [1]:
from langchain_community.document_loaders import ( PyPDFLoader, PDFMinerLoader, PyMuPDFLoader, UnstructuredPDFLoader, UnstructuredWordDocumentLoader)
from langchain_core.documents import Document
from typing import List,Dict,Any

import transformers
import spacy
import re
import os
import google.generativeai as genai
from collections import defaultdict
import pdfplumber
# from transformers.utils.import_utils import candidates
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')

F:\rag_practice_repo\rag_practice_repo\DATA_PARSING_INGESTION\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
F:\rag_practice_repo\rag_practice_repo\DATA_PARSING_INGESTION\.venv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
nlp=spacy.load('en_core_web_sm')

In [3]:
try:
    pypdf_loader=PyPDFLoader("data/Resume_Salil_Yadav.pdf")
    pypdf_load=pypdf_loader.load()
    print(pypdf_load)
except Exception as e:
    print(e)

[Document(page_content='                   \n                       Salil Yadav                  \n                                            • Noida, U.P India • salilyadav107@gmail.com • 7007240256 • LinkedIn    • GitHub \n \nAbout Me \n \nMy primary goal is to continually learn, evolve, and apply my skills to contribute to the advancement of organizations and th e \nbetterment of humanity. I am particularly enthusiastic about delving into the realms of R&D in Artificial Intelligence. \n \n \nEducation \nBachelor of Technology Computer Science  \nAMITY UNIVERSITY, Noida, U.P, India -   GPA: 7.91 \n2020-2024 \n• Relevant Coursework: Database Management, Algorithms, Artificial Intelligence, Digital Image Processing, \nPython (Programming Language), Computer Vision, MATLAB. \n• Honors & Awards: qualified for National level Hackathons (UPES) and certificates.  \n• Clubs: GDSC, SEC Club of Amity, Rotaract Club, NGO work Umeed HVCO. \n \nSir Padampat Singhania Education Centre Kanpur U.P 

In [4]:
content  = pypdf_load[0].page_content + pypdf_load[1].page_content

In [5]:
content

'                   \n                       Salil Yadav                  \n                                            • Noida, U.P India • salilyadav107@gmail.com • 7007240256 • LinkedIn    • GitHub \n \nAbout Me \n \nMy primary goal is to continually learn, evolve, and apply my skills to contribute to the advancement of organizations and th e \nbetterment of humanity. I am particularly enthusiastic about delving into the realms of R&D in Artificial Intelligence. \n \n \nEducation \nBachelor of Technology Computer Science  \nAMITY UNIVERSITY, Noida, U.P, India -   GPA: 7.91 \n2020-2024 \n• Relevant Coursework: Database Management, Algorithms, Artificial Intelligence, Digital Image Processing, \nPython (Programming Language), Computer Vision, MATLAB. \n• Honors & Awards: qualified for National level Hackathons (UPES) and certificates.  \n• Clubs: GDSC, SEC Club of Amity, Rotaract Club, NGO work Umeed HVCO. \n \nSir Padampat Singhania Education Centre Kanpur U.P                        

In [6]:
content

'                   \n                       Salil Yadav                  \n                                            • Noida, U.P India • salilyadav107@gmail.com • 7007240256 • LinkedIn    • GitHub \n \nAbout Me \n \nMy primary goal is to continually learn, evolve, and apply my skills to contribute to the advancement of organizations and th e \nbetterment of humanity. I am particularly enthusiastic about delving into the realms of R&D in Artificial Intelligence. \n \n \nEducation \nBachelor of Technology Computer Science  \nAMITY UNIVERSITY, Noida, U.P, India -   GPA: 7.91 \n2020-2024 \n• Relevant Coursework: Database Management, Algorithms, Artificial Intelligence, Digital Image Processing, \nPython (Programming Language), Computer Vision, MATLAB. \n• Honors & Awards: qualified for National level Hackathons (UPES) and certificates.  \n• Clubs: GDSC, SEC Club of Amity, Rotaract Club, NGO work Umeed HVCO. \n \nSir Padampat Singhania Education Centre Kanpur U.P                        

In [7]:
def normalize_phrase(s: str) -> str:
    s = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+','[EMAIL]', s)
    s = re.sub(r'\+?\d[\d\s\()]{7,}\d', '[PHONE]', s)
    # remove URLs (LinkedIn, GitHub, etc.)
    s = re.sub(r'https?://\S+|www\.\S+', '[URL]', s)
    return s.strip()

content = normalize_phrase(content)


In [8]:
content

'Salil Yadav                  \n                                            • Noida, U.P India • [EMAIL] • [PHONE] • LinkedIn    • GitHub \n \nAbout Me \n \nMy primary goal is to continually learn, evolve, and apply my skills to contribute to the advancement of organizations and th e \nbetterment of humanity. I am particularly enthusiastic about delving into the realms of R&D in Artificial Intelligence. \n \n \nEducation \nBachelor of Technology Computer Science  \nAMITY UNIVERSITY, Noida, U.P, India -   GPA: 7.91 \n2020-2024 \n• Relevant Coursework: Database Management, Algorithms, Artificial Intelligence, Digital Image Processing, \nPython (Programming Language), Computer Vision, MATLAB. \n• Honors & Awards: qualified for National level Hackathons (UPES) and certificates.  \n• Clubs: GDSC, SEC Club of Amity, Rotaract Club, NGO work Umeed HVCO. \n \nSir Padampat Singhania Education Centre Kanpur U.P                                                                                March 2

In [9]:
from openai import OpenAI

client = OpenAI()

resume_text = content
prompt = f"""
    You are an expert HR data extraction assistant.
    You will be provided with a resume and a job description.
    Your task is to extract the relevant information from the resume that is most relevant to the job description.
    The output should be a JSON object with the following structure:
    {{
        "location": "",
        "skills": [],
        "total_experience": "",
        "work_experience": [
            {{
                "title": "",
                "company": "",
                "position": "",
                "start_date": "",
                "end_date": "",
                "description": ""
            }}
        ],
        "education": [
            {{
                "degree": "",
                "institute": "",
                "field_of_study": "",
                "start_date": "",
                "end_date": "",
            }}
        ],
        "certifications": [],
        "projects": [],
        "interests": [],
    }}
    Rules:
    - The output should be a valid JSON object.
    - Calculate total_experience from start_date to end_date in work_experience.
    - If any fields are missing, leave it as empty.
    - Do not include extra text or summary.
    - Do not include any other information.

    Resume text:
    ---
    {resume_text}
    ---
"""
# prompt = f"""
# You are an expert HR data extraction assistant.
#
# Extract key details from the resume text below and return only a valid JSON object in this format:
#
# {{
#   "name": "",
#   "email": "",
#   "phone": "",
#   "location": "",
#   "total_experience_years": "",
#   "education": [
#     {{"degree": "", "institution": "", "start_year": "", "end_year": ""}}
#   ],
#   "skills": [],
#   "work_experience": [
#     {{"title": "", "company": "", "start_date": "", "end_date": "", "description": ""}}
#   ]
# }}
#
# Rules:
# - Return only in valid JSON.
# - Calculate total work experience experience from work experience.
# - If any field is missing, leave it empty.
# - Do not include extra text or commentary.
#
# Resume text:
# ---
# {resume_text}
# ---
# """


# response = client.chat.completions.create(
#     model="gpt-4o-mini",  # or "gpt-4-turbo" / "gpt-3.5-turbo"
#     messages=[{"role": "user", "content": prompt}],
#     temperature=0
# )
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel("gemini-2.0-flash")  # or "gemini-1.5-pro"

response = model.generate_content(prompt)

print(response.text)
# print(response.choices[0].message.content)


```json
{
    "location": "Noida, U.P India",
    "skills": [
        "Analytical Skills",
        "Quick Learner",
        "C++",
        "Statistics",
        "SQL",
        "TensorFlow",
        "Machine Learning",
        "Team Collaboration",
        "Data Structures",
        "SDLC",
        "Python",
        "Computer Vision",
        "MATLAB",
        "Deep Learning",
        "Neural Network",
        "NLP"
    ],
    "total_experience": "0 years",
    "work_experience": [
        {
            "title": "Research Intern",
            "company": "IIT Patna",
            "position": "Research Intern",
            "start_date": "Feb 2023",
            "end_date": "April 2023",
            "description": "Worked on Vision transformer, deep learning, Neural Network"
        },
        {
            "title": "Machine learning Intern",
            "company": "Ignitus",
            "position": "Machine learning Intern",
            "start_date": "June 2023",
            "end_date": "Ju

In [93]:
json_res = re.sub(r'```json','',response.text)

In [10]:
cleaned_json=response.text.strip('`json \n')

In [11]:
import json
data = json.loads(cleaned_json)
print(data)

{'location': 'Noida, U.P India', 'skills': ['Analytical Skills', 'Quick Learner', 'C++', 'Statistics', 'SQL', 'TensorFlow', 'Machine Learning', 'Team Collaboration', 'Data Structures', 'SDLC', 'Python', 'Computer Vision', 'MATLAB', 'Deep Learning', 'Neural Network', 'NLP'], 'total_experience': '0 years', 'work_experience': [{'title': 'Research Intern', 'company': 'IIT Patna', 'position': 'Research Intern', 'start_date': 'Feb 2023', 'end_date': 'April 2023', 'description': 'Worked on Vision transformer, deep learning, Neural Network'}, {'title': 'Machine learning Intern', 'company': 'Ignitus', 'position': 'Machine learning Intern', 'start_date': 'June 2023', 'end_date': 'June 2023', 'description': 'Worked on PCA Algorithm with EDA on NMIST dataset.'}], 'education': [{'degree': 'Bachelor of Technology', 'institute': 'AMITY UNIVERSITY', 'field_of_study': 'Computer Science', 'start_date': '2020', 'end_date': '2024'}], 'certifications': ['Introduction to Generative AI Learning Path - Google

In [17]:
data.get('projects')

[{'title': 'Liver segmentation using monai to detect tumor (In-house Internship College)',
  'description': 'Worked on healthcare imaging with use of AI (Monai and Pytorch).\nUsed UNet, Image Segmentation, object detection, linear regression, loss function etc in project.\nGot to detect tumor through given CT/MRI scans.'},
 {'title': 'Deep Convolutional Generative Adversarial Network (DCGAN) for Realistic Human Face Generation',
  'description': 'Developed a DCGAN model for generating lifelike human faces.\nUtilized a dataset of real face images and pre-processed them for consistency.\nDesigned and implemented the generator and discriminator architecture.\nTrained the DCGAN iteratively to produce high-quality synthetic images, demonstrating proficiency in GANs and deep learning techniques .'},
 {'title': 'Toxic Comment Classifier using NLP and ML',
  'description': 'This Project aims Towards Classifying English Comments in 6 Different Categories\n(Toxic, Severe Toxic, Threat, Obscene, 